# Week 7 Assessment — Beat the Instructor

**By:** Jaymineh  
**Dataset:** ed-donner/items_lite  
**Model:** Qwen 2.5 3B (QLoRA). *Official Qwen2.5 has 3B/7B; no 4B text-only. Qwen3.5-4B is multimodal.*

Goal: Beat MAE **39.85** on the Price is Right test set.

## 0. Colab: clone the repo first (run this cell once)

If you **uploaded only this notebook** to Colab, the repo is not there—so `pricer` and `util` are missing. Run the cell below to clone the course repo; then run the rest of the notebook.

In [ ]:
# Run this first on Colab if you only uploaded the notebook (fixes "No module named 'pricer'" etc.)
import sys
from pathlib import Path

# Your fork and branch (change if needed)
REPO_URL = "https://github.com/jaymineh/llm_engineering.git"
REPO_BRANCH = "jaymineh"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    repo_dir = Path("/content/llm_engineering")
    if not (repo_dir / "week7" / "pricer").exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(repo_dir)], check=True)
        print(f"Cloned branch '{REPO_BRANCH}'")
    else:
        subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", REPO_BRANCH], check=True, capture_output=True)
        subprocess.run(["git", "-C", str(repo_dir), "checkout", REPO_BRANCH], check=True)
        print(f"Checked out branch '{REPO_BRANCH}'")
    week7_dir = repo_dir / "week7"
    sys.path.insert(0, str(week7_dir))
    print(f"Colab: added {week7_dir} to path")
else:
    # Local: find week7
    week7_dir = Path.cwd()
    while week7_dir.name != "week7" and len(week7_dir.parts) > 1:
        week7_dir = week7_dir.parent
    if week7_dir.name == "week7":
        sys.path.insert(0, str(week7_dir))
    else:
        week7_dir = Path.cwd().parent.parent if (Path.cwd().parent.parent / "pricer").exists() else Path.cwd() / "week7"
        sys.path.insert(0, str(week7_dir))
    print(f"Added to path: {week7_dir}")

## 1. Setup & path

Add week7 to path so we can import `pricer` and `util`.

In [ ]:
# If you ran the Colab cell above, path is already set. Otherwise (local) ensure week7 is on path.
import sys
from pathlib import Path

def _week7_on_path():
    try:
        import pricer.items
        return True
    except ImportError:
        return False

if not _week7_on_path():
    week7_dir = Path.cwd()
    while week7_dir.name != "week7" and len(week7_dir.parts) > 1:
        week7_dir = week7_dir.parent
    if week7_dir.name == "week7":
        sys.path.insert(0, str(week7_dir))
    else:
        week7_dir = Path.cwd().parent.parent if (Path.cwd().parent.parent / "pricer").exists() else Path.cwd() / "week7"
        sys.path.insert(0, str(week7_dir))
    print(f"Added to path: {week7_dir}")
else:
    print("week7 already on path")

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from tqdm.auto import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from datasets import load_dataset, Dataset, DatasetDict
from peft import LoraConfig, PeftModel
from datetime import datetime

load_dotenv(override=True)

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.getenv("HF_TOKEN")

from pricer.items import Item
from util import evaluate

## 2. Constants

Using **items_lite** and **Qwen/Qwen2.5-3B** (official text-only causal LM; no Qwen2.5-4B).

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-3B"
PROJECT_NAME = "price"
HF_USER = os.getenv("HF_USER", "jaymineh")

ITEMS_DATASET = "ed-donner/items_lite"
CUTOFF = 110

RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}-qwen3b-lite"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

EPOCHS = 1
BATCH_SIZE = 16
MAX_SEQUENCE_LENGTH = 96
GRADIENT_ACCUMULATION_STEPS = 1
QUANT_4_BIT = True
LORA_R = 16
LORA_ALPHA = LORA_R * 2
ATTENTION_LAYERS = ["q_proj", "k_proj", "v_proj", "o_proj"]
MLP_LAYERS = ["gate_proj", "up_proj", "down_proj"]
TARGET_MODULES = ATTENTION_LAYERS
LORA_DROPOUT = 0.1
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = "cosine"
VAL_SIZE = 500
LOG_STEPS = 5
SAVE_STEPS = 100
LOG_TO_WANDB = False
# Cap steps so training finishes within a short Colab session (~30 min at 0.33 it/s). Set to None for full run.
MAX_STEPS = 600

capability = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
use_bf16 = capability[0] >= 8

## 3. HuggingFace login

In [ ]:
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=True)
    print("HuggingFace login OK")
else:
    print("Set HF_TOKEN (env or Colab secret)")

## 4. Load items_lite and build prompt dataset

In [ ]:
train_items, val_items, test_items = Item.from_hub(ITEMS_DATASET)
print(f"Train: {len(train_items):,} | Val: {len(val_items):,} | Test: {len(test_items):,}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

for item in tqdm(train_items + val_items, desc="Train/Val prompts"):
    item.make_prompts(tokenizer, CUTOFF, True)
for item in tqdm(test_items, desc="Test prompts"):
    item.make_prompts(tokenizer, CUTOFF, False)

def to_datapoint(item):
    return {"prompt": item.prompt, "completion": item.completion}

train_dp = [to_datapoint(i) for i in train_items]
val_dp = [to_datapoint(i) for i in val_items]
test_dp = [to_datapoint(i) for i in test_items]

train_dataset = Dataset.from_list([{**d, "text": d["prompt"] + d["completion"]} for d in train_dp])
val_dataset = Dataset.from_list([{**d, "text": d["prompt"] + d["completion"]} for d in val_dp])
test_dataset = Dataset.from_list(test_dp)

val_dataset = val_dataset.select(range(min(VAL_SIZE, len(val_dataset))))
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print("Sample prompt:", train_items[0].prompt[:200], "...")
print("Sample completion:", train_items[0].completion)

## 5. Training (QLoRA)

Run on Colab with GPU (T4 or better). Skip this section if you already have a fine-tuned model on the Hub.

In [ ]:
# Install training dependencies (run once per Colab session)
%pip install -q trl bitsandbytes

In [ ]:
try:
    from trl import SFTTrainer, SFTConfig
except ImportError:
    raise ImportError("Install trl: pip install trl")

# Use float16 for training to avoid GradScaler bf16 error on Colab
if QUANT_4_BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )
else:
    quant_config = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=torch.float16,
    )

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id
print(f"Model memory: {base_model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

train_args = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS if MAX_STEPS else -1,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim="paged_adamw_32bit",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    warmup_steps=max(1, int((len(train_dataset) * EPOCHS) / (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS) * WARMUP_RATIO)),
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else "none",
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    dataset_text_field="text",
    save_strategy="steps",
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    push_to_hub=False,
    dataloader_num_workers=2,
)

trainer = SFTTrainer(
    model=base_model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=lora_config,
    args=train_args,
)
trainer.train()
trainer.save_model(PROJECT_RUN_NAME)
print(f"Saved to {PROJECT_RUN_NAME}")

tokenizer.save_pretrained(PROJECT_RUN_NAME)

Optional: push adapter to Hub (set `HF_USER` to your username first):
```python
# trainer.model.push_to_hub(HUB_MODEL_NAME, private=True)
# tokenizer.push_to_hub(HUB_MODEL_NAME)
```

In [ ]:
# Push adapter and tokenizer to Hugging Face Hub (run after training)
# Set HF_USER to your username in Constants (Section 2) and run login (Section 3) first
trainer.model.push_to_hub(HUB_MODEL_NAME, private=True)
tokenizer.push_to_hub(HUB_MODEL_NAME)
print(f"Pushed to https://huggingface.co/{HUB_MODEL_NAME}")

## 6. Evaluation

Load the fine-tuned model (from local save or Hub) and run the evaluator.

In [ ]:
USE_LOCAL_MODEL = True
LOCAL_MODEL_PATH = PROJECT_RUN_NAME
HUB_MODEL_ID = None
HUB_REVISION = None

if USE_LOCAL_MODEL and os.path.isdir(LOCAL_MODEL_PATH):
    model_path = LOCAL_MODEL_PATH
    from_peft = True
    print(f"Using local: {model_path}")
else:
    model_path = HUB_MODEL_ID or HUB_MODEL_NAME
    from_peft = True
    print(f"Using Hub: {model_path}@{HUB_REVISION or 'main'}")

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
)

if from_peft:
    fine_tuned_model = PeftModel.from_pretrained(base_model, model_path)
else:
    fine_tuned_model = base_model

fine_tuned_model.eval()
print("Model loaded.")

In [ ]:
def model_predict(item):
    inputs = tokenizer(item["prompt"], return_tensors="pt").to(fine_tuned_model.device)
    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)

set_seed(42)
evaluate(model_predict, test_dataset)